# 演習4. 待ち合わせ ―― 「まだ来ない」をどう待つか

## シーン

演習3で、キューを鍵で守れば安全に共有できることが分かりました。
しかしパイプラインには、まだ大きな問題が残っています。

```
[ 作る係 ] ──▶ [ キュー ] ──▶ [ 受け取る係 ]
```

**受け取る係が「キューを見に行ったら、まだ空だった」とき、どうすればよいでしょうか。**

これは並行処理でいちばん頻繁に出てくる場面です。この演習では、
素朴なやり方から順に3通り試して、最後に **`std::condition_variable`** にたどり着きます。

測るのは、毎回**同じ2つの数字**です。

- **空かどうか確認した回数** … 無駄働きの量。少ないほどよい
- **平均の遅れ** … データが入ってから取り出されるまでの時間。短いほどよい

この2つが**同時に良くなる方法**を探す、というのがこの演習のすべてです。

## 4-1. 【予測クイズ】ひたすら確認し続ける

いちばん素直なやり方は「空でなくなるまで、何度でも見に行く」です。

```cpp
while (true) {
    std::lock_guard<std::mutex> g(mtx);
    if (!q.empty()) { ... 取り出す ...; break; }
}      // ← 空だったら、すぐまた見に行く
```

これを **ビジーウェイト**（busy wait / スピン）と呼びます。

次のプログラムは、作る係が **1個作るのに約200ms** かけ、それを5個作ります。
受け取る係はビジーウェイトで待ちます。

**実行する前に予測してください。**

- 全体の所要時間は何 ms くらいになるでしょうか
- 「空かどうか確認した回数」は何回くらいになるでしょうか
  （10回？ 1,000回？ それとももっと？）

In [ ]:
%%writefile ex04a.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <chrono>
using namespace std::chrono;

std::queue<int> q;
std::mutex mtx;
long checks = 0;                       // 「空かどうか確認した」回数

// 作る側：200ms かけて1個作り、キューに入れる
void producer() {
    for (int i = 1; i <= 5; i++) {
        std::this_thread::sleep_for(milliseconds(200));
        std::lock_guard<std::mutex> g(mtx);
        q.push(i);
    }
}

// 受け取る側：キューが空でなくなるまで、ひたすら確認し続ける
void consumer() {
    for (int n = 0; n < 5; n++) {
        int v;
        while (true) {
            std::lock_guard<std::mutex> g(mtx);
            checks++;
            if (!q.empty()) { v = q.front(); q.pop(); break; }
        }
        std::cout << "受け取った : " << v << "\n";
    }
}

int main() {
    auto t0 = steady_clock::now();
    std::thread p(producer), c(consumer);
    p.join(); c.join();
    std::cout << "\n所要時間           = "
              << duration_cast<milliseconds>(steady_clock::now() - t0).count() << " ms\n";
    std::cout << "空かどうか確認した回数 = " << checks << " 回\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex04a.cpp -o ex04a && ./ex04a

### 結果 ―― 答えは合っているが、とんでもなく無駄

所要時間は約 1000ms。5個 × 200ms なので、これは正しい値です。
**答えとしては何も間違っていません。**

問題は確認回数です。**数千万回**になったはずです（環境によって桁は変わります）。

1秒間に数千万回、受け取る係は「まだ？」「まだ？」と聞き続けていたことになります。
その間ずっと **CPU を1個まるごと使い切っています**。

```
作る係     ....作る200ms....入れる ....作る200ms....入れる
受け取る係 ????????????????取った????????????????取った
           ↑ この「?」の全部が無駄働き
```

これが困る理由は、CPU が熱くなるからではありません。

- **そのコアを他のスレッドが使えなくなります。** 演習1で見たとおり、
  同時に計算できる数はコア数までです。1本が無駄にコアを占有すれば、
  本当に仕事をしたいスレッドの取り分が減ります
- パイプラインの段が4つあれば、待っている段が3つ。**コアが足りるはずがありません**

> **待っているスレッドが CPU を使ってはいけない。**

これがこの演習の出発点です。

## 4-2. 少し眠りながら確認する（ポーリング）

無駄働きを減らす、いちばん簡単な手当ては「毎回すぐ聞き直さず、少し眠ってから聞く」です。

```cpp
while (true) {
    { std::lock_guard<std::mutex> g(mtx);
      if (!q.empty()) { ... 取り出す ...; break; } }
    std::this_thread::sleep_for(milliseconds(10));   // ← 少し眠る
}
```

これを **ポーリング**（polling）と呼びます。
眠っている間、そのスレッドは CPU を使いません。

では、何 ms 眠るのがよいでしょうか。1ms・10ms・100ms の3通りで測ります。

なお、この版では**遅れ**も測ります。データがキューに入った時刻を一緒に持たせておき、
取り出したときとの差を見ます。

**実行する前に予測してください。** 眠る時間を長くすると、2つの数字はそれぞれどうなるでしょうか。

In [ ]:
%%writefile ex04b.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <chrono>
using namespace std::chrono;

struct Item { int v; steady_clock::time_point born; };   // born = キューに入れた時刻

std::queue<Item> q;
std::mutex mtx;
long checks;                 // 空かどうか確認した回数
long long delay_us;          // 「入れてから取り出すまで」の合計(マイクロ秒)

void producer() {
    for (int i = 1; i <= 5; i++) {
        std::this_thread::sleep_for(milliseconds(200 + 17 * i));   // 作る時間は毎回少し違う
        std::lock_guard<std::mutex> g(mtx);
        q.push({i, steady_clock::now()});
    }
}

void consumer(int nap_ms) {
    for (int n = 0; n < 5; n++) {
        while (true) {
            {
                std::lock_guard<std::mutex> g(mtx);
                checks++;
                if (!q.empty()) {
                    Item it = q.front(); q.pop();
                    delay_us += duration_cast<microseconds>(steady_clock::now() - it.born).count();
                    break;
                }
            }
            std::this_thread::sleep_for(milliseconds(nap_ms));   // ← 少し眠ってから確認し直す
        }
    }
}

void run(int nap_ms) {
    checks = 0; delay_us = 0;
    while (!q.empty()) q.pop();
    std::thread p(producer), c(consumer, nap_ms);
    p.join(); c.join();
    std::cout << nap_ms << " ms 眠る : 確認 " << checks << " 回 , "
              << "平均の遅れ " << (delay_us / 5 / 1000.0) << " ms\n";
}

int main() {
    run(1);
    run(10);
    run(100);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex04b.cpp -o ex04b && ./ex04b

### 結果 ―― 片方を良くすると、もう片方が悪くなる

だいたい次のようになったはずです。

```
   眠る時間     確認回数      平均の遅れ
------------------------------------------
     1 ms       約 1,100回     約  1 ms
    10 ms       約   130回     約  5 ms
   100 ms       約    18回     約 60 ms
```

- **短く眠る** … すぐ気づけるが、確認回数が多い（無駄働きが多い）
- **長く眠る** … 無駄働きは減るが、データが来てから気づくまで遅れる

**この2つは、この方法では両立しません。** どこかで折り合いをつけるしかない ――
ように見えます。

しかも、折り合いの「正解」は決められません。眠る時間を「作る時間の平均」に
合わせればよさそうに思えますが、現実には作る時間はばらつきますし、
そもそも**作る側の都合を受け取る側が知っているとはかぎりません**。

さらに悪いことに、**ポーリングは「たまたま動いてしまう」**という性質があります。
遅れが 60ms あっても、プログラムは正しく動いて終わります。
測らないかぎり誰も気づきません。

> **ポーリングは「間隔をいくつにしても中途半端」。**

必要なのは、**眠る時間を決めなくてよい方法**です。

## 4-3. `condition_variable` ―― 起こしてもらう

発想を変えます。受け取る係が自分で見に行くのをやめて、
**入れた側に起こしてもらう**ことにします。

これを担当するのが **`std::condition_variable`**（条件変数）です。
名前は難しそうですが、やることは2つだけです。

- **`wait(...)`** … 条件が満たされるまで眠る（待つ側が呼ぶ）
- **`notify_one()`** … 待っている人を1人起こす（変化させた側が呼ぶ）

使い方の形は決まっています。

```cpp
std::mutex mtx;
std::condition_variable can_pop;

// 入れる側
{
    std::lock_guard<std::mutex> g(mtx);
    q.push(v);
}
can_pop.notify_one();                              // 起こす

// 取り出す側
std::unique_lock<std::mutex> lk(mtx);              // lock_guard ではダメ
can_pop.wait(lk, [] { return !q.empty(); });       // 条件が満たされるまで眠る
v = q.front(); q.pop();
```

`wait` の2つ目の引数（`[] { return !q.empty(); }`）を **述語（predicate）** と呼びます。
「これが `true` になるまで待つ」という条件です。ラムダ式です
（C++問題集の**問7**で扱いました）。

**実行する前に予測してください。** 確認回数と平均の遅れは、それぞれいくつになるでしょうか。

In [ ]:
%%writefile ex04c.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

struct Item { int v; steady_clock::time_point born; };

std::queue<Item> q;
std::mutex mtx;
std::condition_variable can_pop;      // 「取り出せるようになった」を知らせる係
long checks = 0;
long long delay_us = 0;

void producer() {
    for (int i = 1; i <= 5; i++) {
        std::this_thread::sleep_for(milliseconds(200 + 17 * i));
        {
            std::lock_guard<std::mutex> g(mtx);
            q.push({i, steady_clock::now()});
        }
        can_pop.notify_one();                       // ← 待っている人を1人起こす
    }
}

void consumer() {
    for (int n = 0; n < 5; n++) {
        std::unique_lock<std::mutex> lk(mtx);       // ← unique_lock でないといけない
        can_pop.wait(lk, [] { checks++; return !q.empty(); });
        Item it = q.front(); q.pop();
        delay_us += duration_cast<microseconds>(steady_clock::now() - it.born).count();
        lk.unlock();
        std::cout << "受け取った : " << it.v << "\n";
    }
}

int main() {
    auto t0 = steady_clock::now();
    std::thread p(producer), c(consumer);
    p.join(); c.join();
    std::cout << "\n所要時間           = "
              << duration_cast<milliseconds>(steady_clock::now() - t0).count() << " ms\n";
    std::cout << "空かどうか確認した回数 = " << checks << " 回\n";
    std::cout << "平均の遅れ         = " << (delay_us / 5 / 1000.0) << " ms\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex04c.cpp -o ex04c && ./ex04c

### 結果 ―― 両方いっぺんに良くなった

- 確認回数 = **10回**（5個 × 2回。眠る前に1回、起こされてから1回）
- 平均の遅れ = **1ms未満**
- 所要時間は今までと同じ

ポーリングでは選べなかった「無駄働きゼロ」と「遅れゼロ」が、同時に手に入りました。

### `wait` が実際にやっていること

`can_pop.wait(lk, 述語)` は、1行に見えて**3つの仕事**をしています。

1. 述語を確かめる。**もう `true` なら、眠らずにそのまま先へ進む**
2. `false` なら、**鍵を開けてから**眠る
3. 起こされたら、**鍵を取り直してから**述語をもう一度確かめる。
   まだ `false` なら、また 2 に戻る

**2番目が肝心です。** 鍵を握ったまま眠ったら、入れる側が鍵を取れず、
永久に誰もキューに入れられません。だから待つ前に鍵を手放します。

そして、鍵を手放した以上、起きたあとは取り直さないと先へ進めません。
これが演習3の発展課題4で見た話です。

**`lock_guard` ではなく `unique_lock` を使う理由がこれです。**
途中で開け閉めできるのは `unique_lock` だけでした（3-3）。

### `notify_one` と `notify_all`

- `notify_one()` … 待っている人を**1人**起こす
- `notify_all()` … 待っている人を**全員**起こす

キューに1個入れたのなら、動けるのは1人だけです。全員起こしても、
1人以外は「まだ空だった」と分かってまた眠るだけで、無駄になります。
**1個入れたら `notify_one`** が基本です。

なお、`notify` は**待っている人がいなければ何も起きません**。
「先に notify してしまったせいで通知が消える」ということを心配する必要はありません
―― ただし、それは**述語を書いている場合だけ**です。次でその話をします。

## 4-4. 述語（predicate）はなぜ要るのか

`wait` には述語なしの形もあります。

```cpp
can_pop.wait(lk);                                // 述語なし：起こされたら必ず進む
can_pop.wait(lk, [] { return !q.empty(); });     // 述語あり：条件が真になるまで進まない
```

述語なしのほうが短くて分かりやすく見えます。**なぜわざわざ書くのでしょうか。**

理由は「**起こされたからといって、条件が満たされているとはかぎらない**」からです。
起こす側と起きる側のあいだには時間差があり、その間に他のスレッドが
先に取っていってしまうことがあります。

次のプログラムは、**受け取る係を2人**にして、その様子を見ます。
1個しか入れていないのに、2人とも起きてしまう場面を作ります。

**実行する前に予測してください。** 述語なしの場合、2人目の消費者はどうなるでしょうか。

In [ ]:
%%writefile ex04d.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <string>
#include <chrono>
using namespace std::chrono;

std::queue<int> q;
std::mutex mtx;
std::condition_variable can_pop;
std::mutex cout_mtx;                       // 表示が混ざらないようにするためだけの鍵

void say(const std::string& s) {
    std::lock_guard<std::mutex> g(cout_mtx);
    std::cout << "  " << s << "\n";
}

// 200ms後に1個、さらに300ms後にもう1個入れる
void producer() {
    for (int i = 1; i <= 2; i++) {
        std::this_thread::sleep_for(milliseconds(i == 1 ? 200 : 300));
        {
            std::lock_guard<std::mutex> g(mtx);
            q.push(i);
        }
        say("作った : " + std::to_string(i) + " を入れて、待っている人を起こす");
        can_pop.notify_all();
    }
}

void consumer(int id, bool use_predicate) {
    std::unique_lock<std::mutex> lk(mtx);
    if (use_predicate) can_pop.wait(lk, [] { return !q.empty(); });   // 述語あり
    else               can_pop.wait(lk);                             // 述語なし

    if (q.empty()) {
        say("消費者" + std::to_string(id) + " : 起こされたのに、キューは空だった！");
    } else {
        int v = q.front(); q.pop();
        say("消費者" + std::to_string(id) + " : " + std::to_string(v) + " を取り出した");
    }
}

void run(bool use_predicate) {
    while (!q.empty()) q.pop();
    std::cout << (use_predicate ? "\n【述語あり】 can_pop.wait(lk, []{ return !q.empty(); });\n"
                                : "\n【述語なし】 can_pop.wait(lk);\n");
    std::thread c0(consumer, 0, use_predicate), c1(consumer, 1, use_predicate);
    std::this_thread::sleep_for(milliseconds(50));      // 2人が待ちに入るのを待ってから作り始める
    std::thread p(producer);
    c0.join(); c1.join(); p.join();
    std::cout << "  → キューに残った個数 = " << q.size() << "\n";
}

int main() {
    run(false);
    run(true);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex04d.cpp -o ex04d && ./ex04d

### 結果 ―― 述語がないと、空のキューから取り出そうとする

述語なしのほうは、こうなったはずです。

```
消費者0 : 1 を取り出した
消費者1 : 起こされたのに、キューは空だった！
→ キューに残った個数 = 1
```

被害は2つあります。

1. **消費者1 は空のキューを触ろうとしました。**
   このプログラムでは `if (q.empty())` で確認しているので助かっていますが、
   確認せずに `q.front()` を呼んでいたら、そこで壊れます
2. **2個目のデータが誰にも取り出されず、キューに残りました。**
   消費者1 はもう帰ってしまったからです

述語ありのほうは、消費者1 は1個目の通知で起こされても
「まだ空だ」と分かって**眠り直し**、2個目でちゃんと起きています。

### なぜ空になっていたのか ―― 通知から鍵を取り直すまでの隙間

ここで「通知が届くのが遅れて、データがまだ来ていなかったのだろうか」と考えたくなります。
**違います。データは通知の時点で、すでにキューに入っていました。**

起きたのは「まだ来ていない」ではなく「**もう持っていかれた**」です。順に追います。

なお、以下で「鍵」と書いているのは、すべて**キューを守っている `mtx` ただ1本**のことです。
作る側が `q.push()` のときに取るのも、消費者が `unique_lock lk(mtx)` で取るのも、
`can_pop.wait(lk, ...)` が中で放したり取り直したりするのも、同じこの鍵です。
`condition_variable` は自分専用の鍵を持っていません
（だから `wait` に `lk` を渡す必要があります）。

```
   P = 作る側    C0 = 消費者0    C1 = 消費者1
   鍵 = mtx（キューを守っている1本の鍵）

 1  C0, C1 : wait の中で眠っている
 2  P      : 鍵を取る
 3  P      : q.push(1)                       <- データはこの時点でもう在る
 4  P      : 鍵を放す
 5  P      : notify_all()  ---> C0 と C1 の両方が「起きてよい」印を受け取る
 6  C0     : 鍵を取りに行く -> 取れた
    C1     : 鍵を取りに行く -> C0 が持っているので取れない。待つ
 7  C0     : wait から戻る
 8  C0     : q.front() で 1 を取り出す        <- ここでデータが無くなる
 9  C0     : 鍵を放す
10  C1     : やっと鍵が取れた -> wait から戻る
11  C1     : q は空
```

**5 から 10 のあいだが「隙間」です。** ここで押さえておくことが2つあります。

**① `notify` は、データも鍵も渡さない**

通知は郵便物の配達ではありません。「起きて、自分で確かめに行け」という合図だけです。
データは共有のキューに置いてあるままなので、**先に鍵を取った人が持っていけます。**

**② 起きた人は、鍵を取り直すまで何もできない**

4-3 で見た `wait` の3つの仕事のうち、「起こされたら**鍵を取り直してから**確かめ直す」の部分です。
鍵は一度に1人しか持てないので、2人が起きれば必ず順番待ちになります。
そして待っているあいだに、先に入った人が状態を変えてしまいます。

つまり ――

> **`wait` が返った瞬間に保証されているのは「鍵を持っている」ことだけ。**
> **「条件が成り立っている」ことは保証されていない。**

述語つきの `wait` は、鍵を取り直した直後にもう一度条件を確かめ、
偽ならまた眠ってくれます。だから安全なのです。

### 条件が偽になっている原因は、1つではない

上で見た「横取り」以外にも、起きたときに条件が偽になっている原因があります。

- **通知した数より、待っている人のほうが多い。**
  1個しか入れていないのに `notify_all()` で10人起こせば、9人は必ず外れます（発展課題2）
- **そもそも待っていなかった人に取られる。**
  `wait` で眠っていない別の消費者が、たまたまその瞬間に取りに来て鍵を取れば、
  起こされた人より先に持っていけます
- **誰も通知していないのに起きる。** これは次の節で扱います

**どれも「起きた ≠ 条件が成り立っている」という同じ形**なので、
述語を1つ書いておけば、まとめて片付きます。

### 述語は `while` と同じこと

述語つきの `wait` は、次のコードとまったく同じ意味です。

```cpp
while (!(条件)) {
    can_pop.wait(lk);      // 起こされても、条件を満たしていなければ もう一度待つ
}
```

`if` ではなく **`while`** であることが本質です。「起こされた → 条件を確かめ直す」を
必ず繰り返します。述語つきの形は、この `while` を短く書けるようにしただけのものです。

### もう1つの理由 ―― 誰も起こしていないのに起きることがある

`condition_variable` は、**誰も `notify` していないのに `wait` から戻ってくることがあります。**
これを **spurious wakeup（見かけ上の目覚め）** と呼びます。
OS やハードウェアの都合で起きる現象で、バグではなく仕様です。

述語を書いておけば、この場合も「条件が偽なので眠り直す」となり、何も問題は起きません。

> **`wait` には必ず述語を書く。例外はない。**

これは並行プログラミングで数少ない「常に守ってよい規則」の1つです。

## まとめ

3つのやり方を並べます。

- **ビジーウェイト** … 遅れは最小。ただし CPU を1コア食いつぶす。実用にならない
- **ポーリング** … 眠る時間で「無駄働き」と「遅れ」を天秤にかける。どちらも中途半端
- **`condition_variable`** … 両方とも最良。書き方は決まっているので覚えてしまえばよい

決まった形はこれだけです。

```cpp
// 待つ側
std::unique_lock<std::mutex> lk(mtx);
cv.wait(lk, [] { return 条件; });
...

// 変化させた側
{ std::lock_guard<std::mutex> g(mtx); ...変化させる...; }
cv.notify_one();
```

次の演習5では、この形を使って
**スレッドセーフなキューを最初から最後まで自分で組み立てます。**

## 発展課題

1. ポーリングの眠る時間を「作る側が1個作るのにかかる時間」に合わせれば、
   無駄働きも遅れも小さくできそうに思えます。
   **これがうまくいかないのはどんなときでしょうか。** 2つ以上挙げてください。

2. `notify_one()` を `notify_all()` に変えると、動作は変わるでしょうか。
   受け取る係が **1人のとき**と **10人のとき**で、それぞれ考えてください。
   （10人のときに起きる無駄には名前が付いています）

3. いまのプログラムは「5個受け取ったら終わる」と個数を決め打ちしています。
   では、**作る側が「もう作らない」と伝えたい**とき、受け取る側はどうやって
   `wait` から抜け出せばよいでしょうか。述語をどう書き換えるか考えてください。

4. 述語の中では `q.empty()` という**共有データ**を読んでいます。
   これは競合しないのでしょうか。`wait` が述語をいつ評価しているかを思い出して答えてください。

5. 入れる側の `notify_one()` を、**うっかり書き忘れた**らどうなるでしょうか。
   プログラムは異常終了しますか、それとも別のことが起きますか。